<div style="background: linear-gradient(to right, #000428, #004e92); padding: 50px; border-radius: 30px; text-align: center; color: white; box-shadow: 0 15px 40px rgba(0,0,0,0.6); border: 2px solid #00f2fe;">
    <h1 style="color: #00f2fe; font-size: 4em; margin-bottom: 10px; text-shadow: 4px 4px 15px rgba(0,242,254,0.7);">🚀 Hybrid Semantic Retrieval & Intelligence System (HSRIS)</h1>
    <h2 style="color: #a8e063; letter-spacing: 2px;">NLP Pipeline: Phase 0-6 </h2>
    <hr style="border: 1px solid #00f2fe; width: 40%; margin: 20px auto;">
    <p style="font-size: 1.25em; opacity: 0.9; font-weight: 300;">Advanced Data Science | Dual T4 GPU Optimized | Sparse Neural Architecture</p>
    <div style="margin-top: 20px;">
        <span style="background: rgba(255,255,255,0.1); padding: 8px 15px; border-radius: 20px; font-family: monospace; border: 1px solid #00f2fe;">DS_ASS_03_23F3046_23P3071</span>
    </div>
</div>

<div style='background: linear-gradient(to right, #033333, #004e92); padding: 15px; border-radius: 10px; color: white; margin-top: 30px; border-left: 10px solid #00f2fe;'>
    <h2 style='margin: 0;'>🧠 PHASE 0: SYSTEM ARCHITECTURE & SETUP</h2>
</div>

In [ ]:
import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn
from collections import Counter
import time
import os
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
from ipywidgets import interact, widgets
import warnings
warnings.filterwarnings('ignore')

# Dual GPU Configuration - Dual T4 Optimization
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpu = torch.cuda.device_count()
print(f"🚀 Accelerator: {device} | GPUs available: {n_gpu}")

In [ ]:
def find_file(name, root='/kaggle/input'):
    for r, d, f in os.walk(root):
        if name in f: return os.path.join(r, name)
    return name

csv_path = find_file('customer_support_tickets.csv')
try:
    df = pd.read_csv(csv_path)
    print(f"✅ Dataset Loaded: {len(df)} records.")
except:
    print("⚠️ CSV not found. Loading benchmark sample...")
    df = pd.DataFrame({
        'Ticket Subject': ['login error', 'payment help', 'app crash', 'refund money', 'setup issue'] * 100,
        'Ticket Description': ['cant login fast', 'charged twice help', 'app closing on start', 'want refund now', 'how to setup app'] * 100,
        'Ticket Priority': ['High', 'Low', 'Critical', 'Medium', 'High'] * 100,
        'Ticket Channel': ['Email', 'Chat', 'Phone', 'Email', 'Web'] * 100,
        'Ticket Type': ['Technical', 'Billing', 'Technical', 'Billing', 'General'] * 100
    })

df['text_clean'] = df['Ticket Subject'].fillna('') + " " + df['Ticket Description'].fillna('')

<div style='background: linear-gradient(to right, #1e3c72, #2a5298); padding: 15px; border-radius: 10px; color: white; margin-top: 20px; border-left: 10px solid #a8e063;'>
    <h2 style='margin: 0;'>📊 PHASE 1: CATEGORICAL FOUNDATION (FROM FIRST PRINCIPLES)</h2>
</div>

In [ ]:
# requirement: Map Priority (Low, Medium, High) to ordinal integers
priority_map = {"low": 0, "medium": 1, "high": 2, "critical": 3}
def label_encode_priority(p):
    p_str = str(p).lower()
    return priority_map.get(p_str, -1) # Handles unseen categories with -1

df['priority_encoded'] = df['Ticket Priority'].apply(label_encode_priority)

# Requirement: One-Hot Encoding for Ticket Channel
unique_channels = sorted(df['Ticket Channel'].unique())
channel2idx = {ch: i for i, ch in enumerate(unique_channels)}

def one_hot_channel(ch):
    vec = np.zeros(len(unique_channels))
    if ch in channel2idx: vec[channel2idx[ch]] = 1
    return vec

df['channel_onehot'] = df['Ticket Channel'].apply(one_hot_channel)

# Graph: Priority Distribution
plt.figure(figsize=(10, 4))
sns.countplot(x=df['Ticket Priority'], palette='YlGnBu')
plt.title("Ticket Priority Distribution")
plt.show()
print("✅ First Principles Encoders Initialized.")

<div style='background: linear-gradient(to right, #243b55, #141e30); padding: 15px; border-radius: 10px; color: white; margin-top: 20px; border-left: 10px solid #00f2fe;'>
    <h2 style='margin: 0;'>🔍 PHASE 2: SPARSE REPRESENTATION (KEYWORD RETRIEVAL)</h2>
</div>

In [ ]:
# 2.1 Custom Regex-based Tokenizer
def custom_tokenizer(text):
    text = str(text).lower()
    return re.findall(r'\b[a-z]{2,}\b', text)

# 2.2 N-Gram Generator (Bigrams & Trigrams)
def get_ngrams(tokens, n):
    return ["_".join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

df['tokens'] = df['text_clean'].apply(custom_tokenizer)

# 2.3 Build Vocabulary (Top 5000)
corpus_all = []
for t in df['tokens']:
    corpus_all.extend(t + get_ngrams(t, 2) + get_ngrams(t, 3))

vocab_list = [w for w, _ in Counter(corpus_all).most_common(5000)]
w2idx = {w: i for i, w in enumerate(vocab_list)}
print(f"✅ Multi-Ngram Vocabulary Size: {len(vocab_list)}")

In [ ]:
# 2.4 Manual TF-IDF Implementation
N = len(df)
df_freq = np.zeros(len(vocab_list))
doc_counts = []

for tokens in df['tokens']:
    comb = tokens + get_ngrams(tokens, 2) + get_ngrams(tokens, 3)
    cnt = Counter(comb)
    doc_counts.append(cnt)
    for w in set(comb):
        if w in w2idx: df_freq[w2idx[w]] += 1

idf = np.log((1 + N) / (1 + df_freq)) + 1

# 2.5 Sparse Matrix Construction (Requirement: torch.sparse)
indices, values = [], []
for i, cnt in enumerate(doc_counts):
    for w, c in cnt.items():
        if w in w2idx:
            j = w2idx[w]
            indices.append([i, j])
            values.append(c * idf[j])

indices = torch.LongTensor(indices).t()
values = torch.FloatTensor(values)
tfidf_sparse = torch.sparse_coo_tensor(indices, values, (N, len(vocab_list)))
print(f"✅ TF-IDF Sparse Tensor successfully generated.")

<div style='background: linear-gradient(to right, #004e92, #000428); padding: 15px; border-radius: 10px; color: white; margin-top: 20px; border-left: 10px solid #a8e063;'>
    <h2 style='margin: 0;'>🧠 PHASE 3: DENSE SEMANTIC LAYER (GLOVE EMBEDDINGS)</h2>
</div>

In [ ]:
# 3.1 Load Pre-trained GloVe Embeddings
glove_path = find_file('glove.6B.300d.txt')
dim = 300
UNK_VEC = torch.zeros(dim)

embed_dict = {}
if os.path.exists(glove_path):
    print(f"🔎 Loading GloVe weights from {glove_path}...")
    with open(glove_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            v = line.split()
            embed_dict[v[0]] = torch.tensor([float(x) for x in v[1:]])
            if i > 100000: break
else:
    print("⚠️ GloVe not detected. Loading demo weights...")
    for w in vocab_list[:100]: embed_dict[w] = torch.randn(dim)

# 3.2 Semantic Pooling (Requirement: TF-IDF Weighted Averaging)
def get_doc_vector(tokens):
    v = torch.zeros(dim)
    w_sum = 0
    for t in tokens:
        weight = idf[w2idx[t]] if t in w2idx else 1.0
        v += weight * embed_dict.get(t, UNK_VEC)
        w_sum += weight
    if w_sum > 0: v /= w_sum
    return v

semantic_vecs = torch.stack([get_doc_vector(t) for t in df['tokens']])
print(f"✅ {len(semantic_vecs)} Semantic vectors generated.")

<div style='background: linear-gradient(to right, #1e3c72, #2a5298); padding: 15px; border-radius: 10px; color: white; margin-top: 20px; border-left: 10px solid #00f2fe;'>
    <h2 style='margin: 0;'>⚖️ PHASE 4: HYBRID MULTI-STAGE RETRIEVER</h2>
</div>

In [ ]:
class HSRIS_Retriever(nn.Module):
    def __init__(self, sem_vecs, lex_tfidf):
        super().__init__()
        self.sem_vecs = nn.Parameter(sem_vecs, requires_grad=False)
        self.register_buffer('lex_tfidf', lex_tfidf.to_dense()) 
        
    def forward(self, q_sem, q_lex, alpha=0.4):
        sem_sim = torch.mm(q_sem, self.sem_vecs.T) / (torch.norm(q_sem, dim=1, keepdim=True) * torch.norm(self.sem_vecs, dim=1) + 1e-8)
        lex_sim = torch.mm(q_lex, self.lex_tfidf.T) / (torch.norm(q_lex, dim=1, keepdim=True) * torch.norm(self.lex_tfidf, dim=1) + 1e-8)
        return alpha * lex_sim + (1 - alpha) * sem_sim

model = HSRIS_Retriever(semantic_vecs, tfidf_sparse).to(device)
if n_gpu > 1: model = nn.DataParallel(model)

def multi_search(query, alpha=0.4, k=5):
    tks = custom_tokenizer(query)
    q_s = get_doc_vector(tks).unsqueeze(0).to(device)
    q_l = torch.zeros(1, len(vocab_list)).to(device)
    comb = tks + get_ngrams(tks, 2) + get_ngrams(tks, 3)
    for t in comb:
        if t in w2idx: q_l[0, w2idx[t]] += idf[w2idx[t]]
    
    with torch.no_grad():
        scores = model(q_s, q_l, alpha)
    scores, indices = torch.topk(scores.squeeze(), k)
    return scores.cpu().numpy(), indices.cpu().numpy()

In [ ]:
print(f"✅ Running GPU Performance Test (N=100 Queries)...")
plt.figure(figsize=(10, 4))
batch_sizes_test = [8, 16, 32, 64, 100, 128]
inference_times = []
for b in batch_sizes_test:
    q_sem_b, q_tf_b = torch.randn(b, dim).to(device), torch.randn(b, len(vocab_list)).to(device)
    s = time.time()
    with torch.no_grad(): _ = model(q_sem_b, q_tf_b)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    inference_times.append(time.time() - s)
sns.lineplot(x=batch_sizes_test, y=inference_times, marker='o', color='#00f2fe')
plt.title("Dual T4 GPU Batch Performance")
plt.show()

<div style='background: linear-gradient(to right, #000428, #004e92); padding: 15px; border-radius: 10px; color: white; margin-top: 20px; border-left: 10px solid #a8e063;'>
    <h2 style='margin: 0;'>📊 PHASE 5: EVALUATION (QUANTITATIVE & QUALITATIVE)</h2>
</div>

In [ ]:
print("📈 Evaluating Precision@5 Benchmarks...")

def calculate_precision_at_k(queries_df, k=5):
    p_at_k = []
    # Robust dictionary-based iteration to fix AttributeError
    for q in queries_df.to_dict('records'):
        s, idxs = multi_search(q['text_clean'], alpha=0.4, k=k)
        matches = (df.iloc[idxs]['Ticket Type'] == q['Ticket Type']).sum()
        p_at_k.append(matches / k)
    return np.mean(p_at_k)

sample_test = df.sample(min(100, len(df)))
mean_p5 = calculate_precision_at_k(sample_test)
print(f"\n📜 Final Quantitative Precision@5: {mean_p5*100:.2f}%")

In [ ]:
print("✨ Demonstrating 5 Qualitative Wins Results...")
def compare_search(query):
    s_sem, i_sem = multi_search(query, alpha=0.0, k=1)
    s_lex, i_lex = multi_search(query, alpha=1.0, k=1)
    
    html = f"<div style='background: #112233; color: white; border: 2px solid #00f2fe; padding: 15px; border-radius: 12px; margin-bottom: 25px;'>"
    html += f"<h3 style='color: #00f2fe; margin-top: 0;'>🔎 Query Match: '{query}'</h3>"
    html += "<table style='width: 100%; border-collapse: collapse; font-family: sans-serif;'>"
    html += "<tr><th style='text-align: left; color: #a8e063; padding: 10px; border-bottom: 1px solid #00f2fe;'>🧠 Semantic Layer (GloVe)</th>"
    html += "<th style='text-align: left; color: #00f2fe; padding: 10px; border-bottom: 1px solid #00f2fe;'>🔍 Lexical Layer (TF-IDF)</th></tr>"
    html += f"<tr><td style='padding: 15px; width: 50%;'>{df.iloc[i_sem[0]]['Ticket Subject']}</td>"
    html += f"<td style='padding: 15px; width: 50%; border-left: 1px solid #00f2fe;'>{df.iloc[i_lex[0]]['Ticket Subject']}</td></tr>"
    html += "</table></div>"
    display(HTML(html))

examples = ["wallet issue", "need my cash", "unable to enter", "software closed", "program install"]
for ex in examples: compare_search(ex)

<div style='background: linear-gradient(to right, #0f2027, #203a43, #2c5364); padding: 15px; border-radius: 10px; color: white; margin-top: 20px; border-left: 10px solid #00e4ff;'>
    <h2 style='margin: 0;'>📱 PHASE 6: INTERACTIVE DEPLOYMENT (DASHBOARD & GRADIO)</h2>
</div>

In [ ]:
def style_df(df_in):
    return df_in.style.hide(axis='index')\
                 .set_table_styles([
                     {'selector': 'th', 'props': [('background-color', '#004e92'), ('color', 'white'), ('font-weight', 'bold')]},
                     {'selector': 'tr:hover', 'props': [('background-color', '#eef6ff')]}
                 ])\
                 .background_gradient(subset=['Score'], cmap='Greens')

def hsris_dashboard(query, alpha, k):
    if not query: return
    scores, idxs = multi_search(query, alpha, k=k)
    pred_type = Counter(df.iloc[idxs]['Ticket Type']).most_common(1)[0][0]
    
    display(HTML(f"<div style='background: #000428; border: 2px solid #a8e063; padding: 20px; border-radius: 15px; margin-bottom:10px;'>"
                 f"<h2 style='color: #00f2fe; margin:0;'>🔎 Predictive Insights</h2>"
                 f"<p style='color: white; font-size: 1.25em;'>Recommended Category: <b style='color: #a8e063;'>{pred_type}</b></p></div>"))
    
    results = df.iloc[idxs][['Ticket Subject', 'Ticket Type', 'Ticket Priority']].copy()
    results['Score'] = scores.astype(float)
    display(style_df(results))

interact(hsris_dashboard, 
         query=widgets.Text(value='payment failure', description='Search:'),
         alpha=widgets.FloatSlider(min=0.0, max=1.0, step=0.1, value=0.4, description='Alpha:'),
         k=widgets.IntSlider(min=1, max=10, value=5, description='Show K:'))

In [ ]:
import gradio as gr
def app_logic(query, alpha):
    s, idxs = multi_search(query, alpha, k=3)
    p_type = Counter(df.iloc[idxs]['Ticket Type']).most_common(1)[0][0]
    res = f"### Prediction: **{p_type}**\n---\n"
    for i, idx in enumerate(idxs):
        row = df.iloc[idx]
        res += f"**{i+1}. {row['Ticket Subject']}** (Match: {s[i]:.2f})\nResolution: *{row.get('Resolution', 'Standard Practice')}*\n\n"
    return res

ui = gr.Interface(fn=app_logic, inputs=[gr.Textbox(label="Issue Description"), gr.Slider(0, 1, value=0.4, label="Alpha")],
                  outputs=gr.Markdown(), title="HSRIS AI Support Retriever", theme="soft")
# ui.launch(share=True)